In [1]:
import yaml
import json
import pickle
import pandas as pd
import numpy as np
import itertools
import matplotlib.pyplot as plt
import seaborn as sns
import httpx
import os
import ast
import krippendorff

import warnings
warnings.filterwarnings("ignore")

from prolific import ProlificAPIError, ProlificClient

pd.set_option('display.max_columns', None)

# config = yaml.safe_load(open("config.yaml", "r"))

client = ProlificClient(
    api_token=os.environ["prolific_key"])

In [2]:
ID = "698efef38f6dc0baf73e7f6f"

In [3]:
client.get_study(ID)

{'id': '698efef38f6dc0baf73e7f6f',
 'name': 'Evaluating the Persuasiveness of Debaters',
 'description': '<p>In this study, you will evaluate <strong>persuasiveness in debate transcripts</strong>. You will read transcripts from <strong>two separate debates</strong> on the same topic and judge which speaker is more persuasive.</p><p>Each debate features two participants, Speaker A and Speaker B. Although the debates involve different individuals, <strong>Speaker A represents the same ideological or political position in both debates</strong>.</p><p>Your task is to compare the persuasiveness of <strong>Speaker A across the two debates</strong>, focusing only on how effectively they argue their position.</p><p>There are <strong>no right or wrong answers</strong>! We are interested in your honest, subjective judgment based on the debate content.</p><p></p>',
 'total_available_places': 3234,
 'reward': 120,
 'average_reward_per_hour': 1155.0,
 'external_study_url': 'https://tars.dcp.prod.aw

In [4]:
responses = client.list_submissions(study=ID, page_size=1000)
print(len(responses["results"]))
print(json.dumps(responses['results'][0], indent=2))

1000
{
  "id": "698f546d256d83fbae917ec5",
  "participant_id": "673b7a139cd7db8678fafdcd",
  "started_at": "2026-02-13T16:42:42.802000Z",
  "completed_at": "2026-02-13T16:43:55.622000Z",
  "is_complete": true,
  "time_taken": 72,
  "reward": 12000,
  "status": "AWAITING REVIEW",
  "strata": {},
  "study_code": "NOCODE",
  "bonus_payments": [],
  "ip": "194.233.158.124",
  "return_requested": null,
  "has_siblings": false,
  "time_taken_under_auto_approval_threshold": true,
  "dynamic_payment_percentage": null
}


In [5]:
rejected = client.list_submissions(study=ID, page_size=1000, rejected=True)
print(len(rejected["results"]))

5


In [6]:
df = pd.read_csv(
    "data/019c5684-e8e6-737c-9442-07692f856379.csv"
)

In [7]:
df.head()

,DataPoint_ID,Task_Group_ID,Task_Type,Question,Debate Topic,Debate 1,Debate 2,META_SAMPLE1_POLITICAL_POSITION,META_SAMPLE1_SPEAKERA,META_COMPARISONID_1,META_SAMPLE1_SPEAKERB,META_COMPARISON,META_TASK_GROUP_ID,META_COMPARISONID,META_ATTEN_CHECK_PERSUASIVENESS_ANSWER,META_ATTEN_CHECK_CONFIDENCE_ANSWER,META_SAMPLE1_DIALOGUE_CHUNK,META_PAIR_SESSION_IDS,META_SAMPLE2_POLITICAL_POSITION,Annotator1_ID,Annotator1_Response,Annotator1_Timestamp,Annotator2_ID,Annotator2_Response,Annotator2_Timestamp,Annotator3_ID,Annotator3_Response,Annotator3_Timestamp,Annotator4_ID,Annotator4_Response,Annotator4_Timestamp,Annotator5_ID,Annotator5_Response,Annotator5_Timestamp,Annotator6_ID,Annotator6_Response,Annotator6_Timestamp,Annotator7_ID,Annotator7_Response,Annotator7_Timestamp,Annotator8_ID,Annotator8_Response,Annotator8_Timestamp
0,019c5684-b730-70dd-b5f3-ca369d6a622c,412,multiple_choice,"Based only on the debate transcripts, which Sp...",Debate the trade-offs of government interventi...,"Speaker A: ""How can this be done if the opport...","Speaker A: ""Would you concur that this point r...",left,human_9,1154,gpt-4o_base,"('human_9', 'human_11')",412,2064,NaN,NaN,"[{'speaker': 'human_9', 'timestamp': '2026-01-...","('XZV57F', 'OR2XBB')",left,6318acc6273e916b39a95a85,Speaker A from Debate 2,2026-02-13T10:46:55.851Z,579f55486475d400015ab683,Speaker A from Debate 1,2026-02-13T12:06:32.977Z,5e90781c06c6ee000974d087,Speaker A from Debate 1,2026-02-13T19:45:23.867Z,66066cf2a8876c1d1907732a,Speaker A from Debate 1,2026-02-14T20:40:58.661Z,5d07689bad18d40001bb73c1,Speaker A from Debate 1,2026-02-16T15:57:15.704Z,63d13c076dd249ca3a6bffbb,Speaker A from Debate 1,2026-02-18T22:06:51.931Z,5f0c94983314760efb2814c1,Speaker A from Debate 1,2026-02-21T11:14:56.612Z,NaN,NaN,NaN
1,019c5684-b730-70dd-b5f3-ca369d6a622c,412,multiple_choice,How confident are you in your choice?,Debate the trade-offs of government interventi...,"Speaker A: ""How can this be done if the opport...","Speaker A: ""Would you concur that this point r...",left,human_9,1154,gpt-4o_base,"('human_9', 'human_11')",412,2064,NaN,NaN,"[{'speaker': 'human_9', 'timestamp': '2026-01-...","('XZV57F', 'OR2XBB')",left,6318acc6273e916b39a95a85,Neutral,2026-02-13T10:46:55.851Z,579f55486475d400015ab683,Somewhat confident,2026-02-13T12:06:32.978Z,5e90781c06c6ee000974d087,Very confident,2026-02-13T19:45:23.867Z,66066cf2a8876c1d1907732a,Somewhat unsure,2026-02-14T20:40:58.661Z,5d07689bad18d40001bb73c1,Neutral,2026-02-16T15:57:15.704Z,63d13c076dd249ca3a6bffbb,Very confident,2026-02-18T22:06:51.932Z,5f0c94983314760efb2814c1,Somewhat confident,2026-02-21T11:14:56.612Z,NaN,NaN,NaN
2,019c5683-e116-71d8-8aba-b35b807dd373,173,multiple_choice,"Based only on the debate transcripts, which Sp...",(ATTENTION CHECK: ignore this datapoint and se...,(ATTENTION CHECK: ignore this datapoint and se...,(ATTENTION CHECK: ignore this datapoint and se...,left,human_11,1128,human_4,"('human_11', 'human_8')",173,867,Both Speaker As were equally persuasive,Neutral,"[{'speaker': 'human_11', 'timestamp': '2026-01...","('WTXVCV', 'RSQGNQ')",left,695c0670de1faedbce882831,Both Speaker As were equally persuasive,2026-02-13T10:51:45.569Z,5e8f8ebc09b58f28c329da6f,Both Speaker As were equally persuasive,2026-02-13T14:09:52.885Z,60e41e3e8c52dca4b19a3487,Both Speaker As were equally persuasive,2026-02-13T18:49:51.705Z,6978e5020a6d1cdb00d60fe4,Both Speaker As were equally persuasive,2026-02-14T18:01:04.033Z,64784c2bc85267b924936813,Both Speaker As were equally persuasive,2026-02-16T13:42:10.831Z,60fc3a1e0eb578aa02590e27,Both Speaker As were equally persuasive,2026-02-18T18:46:00.749Z,5f1e8aa76210d9000c23deb5,Both Speaker As were equally persuasive,2026-02-21T08:28:37.218Z,NaN,NaN,NaN
3,019c5683-e116-71d8-8aba-b35b807dd373,173,multiple_choice,How confident are you in your choice?,(ATTENTION CHECK: ignore this datapoint and se...,(ATTENTION CHECK: ignore this datapoint and se...,(ATTENTION CHECK: ignore this datapoint and se...,left,human_11,1128,h

In [8]:
comparison_ids = df["META_COMPARISONID"].unique().tolist()

annotator_cols = []
for col in df.columns:
    if col.startswith("Annotator") and col.endswith('_ID'):
        base = col.replace("_ID", "")
        annotator_cols.append((
            col,                          # ID column
            base + "_Timestamp",          # Timestamp column
            base + "_Response"            # Response column
        ))
annotator_cols

[('Annotator1_ID', 'Annotator1_Timestamp', 'Annotator1_Response'),
 ('Annotator2_ID', 'Annotator2_Timestamp', 'Annotator2_Response'),
 ('Annotator3_ID', 'Annotator3_Timestamp', 'Annotator3_Response'),
 ('Annotator4_ID', 'Annotator4_Timestamp', 'Annotator4_Response'),
 ('Annotator5_ID', 'Annotator5_Timestamp', 'Annotator5_Response'),
 ('Annotator6_ID', 'Annotator6_Timestamp', 'Annotator6_Response'),
 ('Annotator7_ID', 'Annotator7_Timestamp', 'Annotator7_Response'),
 ('Annotator8_ID', 'Annotator8_Timestamp', 'Annotator8_Response')]

In [9]:
rejected_ids = [sub['participant_id'] for sub in rejected['results']]
rejected_ids

['5b70252c99982e000145ccf7',
 '5cd80c45dcabe80001040e88',
 '5d78d57c8f579d001518e74f',
 '5fcfa3168335430d143b4431',
 '67c3321da0a96b0538b4abbe']

In [10]:
attention_df = df[df["META_ATTEN_CHECK_PERSUASIVENESS_ANSWER"].notna() & df["META_ATTEN_CHECK_CONFIDENCE_ANSWER"].notna()]

In [11]:
ids_failed_atten_checks = {sub_id: {} for sub_id in attention_df["Annotator1_ID"].unique()}
for id in ids_failed_atten_checks:
    ids_failed_atten_checks[id]["persuassiveness_question"] = 0
    ids_failed_atten_checks[id]["confidence_question"] = 0

In [12]:
all_annotator_ids = set()
for col_id, col_ts, col_resp in annotator_cols:
    all_annotator_ids.update(attention_df[col_id].dropna().unique())

ids_failed_atten_checks = {sub_id: {} for sub_id in all_annotator_ids}

In [13]:
all_annotator_ids = set()
for col_id, col_ts, col_resp in annotator_cols:
    all_annotator_ids.update(attention_df[col_id].dropna().unique())

ids_failed_atten_checks = {sub_id: {} for sub_id in all_annotator_ids}

for id in ids_failed_atten_checks:
    ids_failed_atten_checks[id]["persuassiveness_question"] = 0
    ids_failed_atten_checks[id]["confidence_question"] = 0

for idx, row in attention_df.iterrows():
    for col_id, col_ts, col_resp in annotator_cols:
        if row[col_id] in rejected_ids or pd.isna(row[col_id]):
            continue

        response = row[col_resp]

        atten_count = 0
        if row["Question"] == "Based only on the debate transcripts, which Speaker A presented the more persuasive arguments?":
            target = row["META_ATTEN_CHECK_PERSUASIVENESS_ANSWER"]

            if target != response:
                ids_failed_atten_checks[df.at[idx, col_id]]["persuassiveness_question"] += 1

        else:
            # Question == 'How confident are you in your choice?'

            target = row["META_ATTEN_CHECK_CONFIDENCE_ANSWER"]

        
            if target != response:
                ids_failed_atten_checks[df.at[idx,col_id]]["confidence_question"] += 1

print(ids_failed_atten_checks)
len(ids_failed_atten_checks)

{'68136160929cc479375d3c20': {'persuassiveness_question': 0, 'confidence_question': 0}, '6979cf9aa50cf9eff6c75149': {'persuassiveness_question': 0, 'confidence_question': 0}, '566d950e57f93300112d0a7b': {'persuassiveness_question': 0, 'confidence_question': 0}, '677ac6f15c4ac72aaffcab23': {'persuassiveness_question': 0, 'confidence_question': 0}, '66d20f7f74d51c4de3cf0cce': {'persuassiveness_question': 0, 'confidence_question': 0}, '6743316520c610ff047fd40a': {'persuassiveness_question': 0, 'confidence_question': 0}, '5fad3af0a272a23e5aef2aa6': {'persuassiveness_question': 0, 'confidence_question': 0}, '654cc4367585dc78f77e81cb': {'persuassiveness_question': 0, 'confidence_question': 0}, '5e3afe879d5f1e30b75b9ca6': {'persuassiveness_question': 0, 'confidence_question': 0}, '5964e073241f8d0001065056': {'persuassiveness_question': 0, 'confidence_question': 0}, '5be742dd0e366e000175c8ba': {'persuassiveness_question': 0, 'confidence_question': 0}, '664a0d8b5ef52adae600c9fa': {'persuassiven

3251

In [15]:
failed_count = 0

for id, att_check in ids_failed_atten_checks.items():
    if att_check["persuassiveness_question"] > 1 and att_check["confidence_question"] > 1:
        failed_count += 1

failed_count

11

In [18]:
df.head()

,DataPoint_ID,Task_Group_ID,Task_Type,Question,Debate Topic,Debate 1,Debate 2,META_SAMPLE1_POLITICAL_POSITION,META_SAMPLE1_SPEAKERA,META_COMPARISONID_1,META_SAMPLE1_SPEAKERB,META_COMPARISON,META_TASK_GROUP_ID,META_COMPARISONID,META_ATTEN_CHECK_PERSUASIVENESS_ANSWER,META_ATTEN_CHECK_CONFIDENCE_ANSWER,META_SAMPLE1_DIALOGUE_CHUNK,META_PAIR_SESSION_IDS,META_SAMPLE2_POLITICAL_POSITION,Annotator1_ID,Annotator1_Response,Annotator1_Timestamp,Annotator2_ID,Annotator2_Response,Annotator2_Timestamp,Annotator3_ID,Annotator3_Response,Annotator3_Timestamp,Annotator4_ID,Annotator4_Response,Annotator4_Timestamp,Annotator5_ID,Annotator5_Response,Annotator5_Timestamp,Annotator6_ID,Annotator6_Response,Annotator6_Timestamp,Annotator7_ID,Annotator7_Response,Annotator7_Timestamp,Annotator8_ID,Annotator8_Response,Annotator8_Timestamp
0,019c5684-b730-70dd-b5f3-ca369d6a622c,412,multiple_choice,"Based only on the debate transcripts, which Sp...",Debate the trade-offs of government interventi...,"Speaker A: ""How can this be done if the opport...","Speaker A: ""Would you concur that this point r...",left,human_9,1154,gpt-4o_base,"('human_9', 'human_11')",412,2064,NaN,NaN,"[{'speaker': 'human_9', 'timestamp': '2026-01-...","('XZV57F', 'OR2XBB')",left,6318acc6273e916b39a95a85,Speaker A from Debate 2,2026-02-13T10:46:55.851Z,579f55486475d400015ab683,Speaker A from Debate 1,2026-02-13T12:06:32.977Z,5e90781c06c6ee000974d087,Speaker A from Debate 1,2026-02-13T19:45:23.867Z,66066cf2a8876c1d1907732a,Speaker A from Debate 1,2026-02-14T20:40:58.661Z,5d07689bad18d40001bb73c1,Speaker A from Debate 1,2026-02-16T15:57:15.704Z,63d13c076dd249ca3a6bffbb,Speaker A from Debate 1,2026-02-18T22:06:51.931Z,5f0c94983314760efb2814c1,Speaker A from Debate 1,2026-02-21T11:14:56.612Z,NaN,NaN,NaN
1,019c5684-b730-70dd-b5f3-ca369d6a622c,412,multiple_choice,How confident are you in your choice?,Debate the trade-offs of government interventi...,"Speaker A: ""How can this be done if the opport...","Speaker A: ""Would you concur that this point r...",left,human_9,1154,gpt-4o_base,"('human_9', 'human_11')",412,2064,NaN,NaN,"[{'speaker': 'human_9', 'timestamp': '2026-01-...","('XZV57F', 'OR2XBB')",left,6318acc6273e916b39a95a85,Neutral,2026-02-13T10:46:55.851Z,579f55486475d400015ab683,Somewhat confident,2026-02-13T12:06:32.978Z,5e90781c06c6ee000974d087,Very confident,2026-02-13T19:45:23.867Z,66066cf2a8876c1d1907732a,Somewhat unsure,2026-02-14T20:40:58.661Z,5d07689bad18d40001bb73c1,Neutral,2026-02-16T15:57:15.704Z,63d13c076dd249ca3a6bffbb,Very confident,2026-02-18T22:06:51.932Z,5f0c94983314760efb2814c1,Somewhat confident,2026-02-21T11:14:56.612Z,NaN,NaN,NaN
2,019c5683-e116-71d8-8aba-b35b807dd373,173,multiple_choice,"Based only on the debate transcripts, which Sp...",(ATTENTION CHECK: ignore this datapoint and se...,(ATTENTION CHECK: ignore this datapoint and se...,(ATTENTION CHECK: ignore this datapoint and se...,left,human_11,1128,human_4,"('human_11', 'human_8')",173,867,Both Speaker As were equally persuasive,Neutral,"[{'speaker': 'human_11', 'timestamp': '2026-01...","('WTXVCV', 'RSQGNQ')",left,695c0670de1faedbce882831,Both Speaker As were equally persuasive,2026-02-13T10:51:45.569Z,5e8f8ebc09b58f28c329da6f,Both Speaker As were equally persuasive,2026-02-13T14:09:52.885Z,60e41e3e8c52dca4b19a3487,Both Speaker As were equally persuasive,2026-02-13T18:49:51.705Z,6978e5020a6d1cdb00d60fe4,Both Speaker As were equally persuasive,2026-02-14T18:01:04.033Z,64784c2bc85267b924936813,Both Speaker As were equally persuasive,2026-02-16T13:42:10.831Z,60fc3a1e0eb578aa02590e27,Both Speaker As were equally persuasive,2026-02-18T18:46:00.749Z,5f1e8aa76210d9000c23deb5,Both Speaker As were equally persuasive,2026-02-21T08:28:37.218Z,NaN,NaN,NaN
3,019c5683-e116-71d8-8aba-b35b807dd373,173,multiple_choice,How confident are you in your choice?,(ATTENTION CHECK: ignore this datapoint and se...,(ATTENTION CHECK: ignore this datapoint and se...,(ATTENTION CHECK: ignore this datapoint and se...,left,human_11,1128,h

## Removing rejected ids from df

In [19]:
comparison_ids = df["META_COMPARISONID"].unique().tolist()

annotator_cols = []
for col in df.columns:
    if col.startswith("Annotator") and col.endswith('_ID'):
        base = col.replace("_ID", "")
        annotator_cols.append((
            col,                          # ID column
            base + "_Timestamp",          # Timestamp column
            base + "_Response"            # Response column
        ))
annotator_cols

[('Annotator1_ID', 'Annotator1_Timestamp', 'Annotator1_Response'),
 ('Annotator2_ID', 'Annotator2_Timestamp', 'Annotator2_Response'),
 ('Annotator3_ID', 'Annotator3_Timestamp', 'Annotator3_Response'),
 ('Annotator4_ID', 'Annotator4_Timestamp', 'Annotator4_Response'),
 ('Annotator5_ID', 'Annotator5_Timestamp', 'Annotator5_Response'),
 ('Annotator6_ID', 'Annotator6_Timestamp', 'Annotator6_Response'),
 ('Annotator7_ID', 'Annotator7_Timestamp', 'Annotator7_Response'),
 ('Annotator8_ID', 'Annotator8_Timestamp', 'Annotator8_Response')]

In [20]:
for idx, row in df.iterrows():
    for col_id, col_ts, col_resp in annotator_cols:
        if row[col_id] in rejected_ids:
            df.at[idx, col_id] = np.nan
            df.at[idx, col_ts] = np.nan
            df.at[idx, col_resp] = np.nan


In [23]:
# Accepting passed submissions and paying participants

participant_IDs = [col for col in df.columns if col.startswith("Annotator") and col.endswith("_ID")]
unique_participant_ids = pd.unique(df[participant_IDs].values.ravel())
unique_participant_ids = [pid for pid in unique_participant_ids if pd.notnull(pid)]

In [25]:
client.bulk_approve_submissions(study_id=ID, participant_ids=unique_participant_ids)

'The request to bulk approve has been made successfully.'

## Krippendorff's alpha

In [26]:
speakers = df["META_COMPARISON"].unique().tolist()

unique_speakers = set()
for s in speakers:
    unique_speakers.add(ast.literal_eval(s)[0])
    unique_speakers.add(ast.literal_eval(s)[1])

unique_speakers =  list(unique_speakers)

In [27]:
len(unique_speakers)

22

In [28]:
df = df[~df["Debate Topic"].str.contains("ATTENTION CHECK:", na=False)]

In [29]:
results_df = df[df["Question"] == "Based only on the debate transcripts, which Speaker A presented the more persuasive arguments?"].copy()
conf_df = df[df["Question"] == "How confident are you in your choice?"].copy()

We need to create a 1386x3234  matrix to compute Krippendorff's alpha

In [30]:
comparison_ids = df["META_COMPARISONID"].unique().tolist()

annotator_ids_cols = []
for col in df.columns:
    if col.startswith("Annotator") and col.endswith('_ID'):
        annotator_ids_cols.append(col)
annotator_ids_cols

['Annotator1_ID',
 'Annotator2_ID',
 'Annotator3_ID',
 'Annotator4_ID',
 'Annotator5_ID',
 'Annotator6_ID',
 'Annotator7_ID',
 'Annotator8_ID']

In [31]:
rejected_sub_ids = [x["id"] for x in rejected["results"]]

In [32]:
rejected_sub_ids

['69925375210c4bcc097cc9d3',
 '69905398aace112378a0ba8b',
 '698f0d2580f565544157acae',
 '69923415fc34882a04f2cc3d',
 '698eff7244b230c546d4f9c6']

In [33]:
annotator_ids = []

for id in annotator_ids_cols:
    annotator_ids.extend(results_df[id].unique().tolist())

for id in rejected_ids:
    if id in annotator_ids:
        annotator_ids.remove(id)

In [34]:
# map annotator ids to index in result mat
annotator_map = {annotator_ids[i]: i for i in range(len(annotator_ids))}

In [35]:
comparison_ids = df["META_COMPARISONID"].unique().tolist()

annotator_cols = []
for col in df.columns:
    if col.startswith("Annotator") and col.endswith('_ID'):
        base = col.replace("_ID", "")
        annotator_cols.append((
            col,                          # ID column
            base + "_Timestamp",          # Timestamp column
            base + "_Response"            # Response column
        ))
annotator_cols

[('Annotator1_ID', 'Annotator1_Timestamp', 'Annotator1_Response'),
 ('Annotator2_ID', 'Annotator2_Timestamp', 'Annotator2_Response'),
 ('Annotator3_ID', 'Annotator3_Timestamp', 'Annotator3_Response'),
 ('Annotator4_ID', 'Annotator4_Timestamp', 'Annotator4_Response'),
 ('Annotator5_ID', 'Annotator5_Timestamp', 'Annotator5_Response'),
 ('Annotator6_ID', 'Annotator6_Timestamp', 'Annotator6_Response'),
 ('Annotator7_ID', 'Annotator7_Timestamp', 'Annotator7_Response'),
 ('Annotator8_ID', 'Annotator8_Timestamp', 'Annotator8_Response')]

In [36]:
annotator_ids = [a for a in annotator_ids if pd.notna(a)]
annotator_ids = list(dict.fromkeys(annotator_ids)) 

In [37]:
annotator_map = {annotator_ids[i]: i for i in range(len(annotator_ids))}

In [38]:
result_mat = []

for idx, row in results_df.iterrows():
    row_results = []
    for j in range(len(annotator_map)):
        row_results.append(None)
    result_mat.append(row_results)

In [39]:
for row_pos, (_, row) in enumerate(results_df.iterrows()):
    for col_id, col_ts, col_resp in annotator_cols:
        if row[col_id] in rejected_ids or pd.isna(row[col_id]):
            continue
        annotator_id = row[col_id]
        mapped_idx = annotator_map.get(annotator_id)
        if mapped_idx is None:
            continue  # avoid KeyError
        result_mat[row_pos][annotator_map[annotator_id]] = row[col_resp]

In [40]:
data = np.asarray(result_mat)

In [41]:
persuasiveness_map = {
    "Speaker A from Debate 1": 1,
    "Both Speaker As were equally persuasive": 2,
    "Speaker A from Debate 2": 3,
}

In [42]:
def map_persuasion(val):
    if val is None or pd.isna(val):
        return np.nan
    return persuasiveness_map.get(val, np.nan)

numeric_data = np.vectorize(map_persuasion, otypes=[float])(data)


In [43]:
# Convert to float and set missing values (if any) to np.nan
numeric_data = numeric_data.astype(float)
numeric_data[pd.isna(data)] = np.nan

In [44]:
alpha = krippendorff.alpha(
    reliability_data=numeric_data.T,
    level_of_measurement="nominal",
)

In [45]:
print(alpha)

0.18823295288395459


### Using value counts

In [46]:
val_counts = np.zeros((len(results_df), 3))
val_counts

array([[0., 0., 0.],
       [0., 0., 0.],
       [0., 0., 0.],
       ...,
       [0., 0., 0.],
       [0., 0., 0.],
       [0., 0., 0.]], shape=(1386, 3))

In [47]:
len(results_df)

1386

In [48]:
val_counts = np.zeros((len(results_df), 3))

for idx, (_, row) in enumerate(results_df.iterrows()):
    for col_id, col_ts, col_resp in annotator_cols:
        if row[col_id] in rejected_ids or pd.isna(row[col_id]):
            continue

        if row[col_resp] == "Speaker A from Debate 1":
            val_counts[idx, 0] += 1
        elif row[col_resp] == "Speaker A from Debate 2":
            val_counts[idx, 1] += 1
        else:
            val_counts[idx, 2] += 1

In [49]:
val_counts[1:20,:]

array([[7., 0., 0.],
       [1., 3., 3.],
       [0., 7., 0.],
       [5., 0., 2.],
       [3., 3., 1.],
       [1., 2., 4.],
       [4., 3., 0.],
       [1., 6., 0.],
       [1., 6., 0.],
       [3., 4., 0.],
       [4., 3., 0.],
       [6., 0., 2.],
       [2., 4., 1.],
       [5., 1., 1.],
       [2., 5., 0.],
       [2., 5., 0.],
       [2., 3., 2.],
       [1., 5., 1.],
       [4., 1., 2.]])

In [50]:
krippendorff.alpha(value_counts=val_counts, level_of_measurement="nominal")

np.float64(0.18823295288395459)

In [146]:
val_counts = np.zeros((len(results_df), 5))
# likert_map = {
#         "Very unsure": 1,
#             "Somewhat unsure": 2,
#                 "Neutral": 3,
#                     "Somewhat confident": 4,
#                         "Very confident": 5
#                         }

for idx, (_, row) in enumerate(conf_df.iterrows()):
    for col_id, col_ts, col_resp in annotator_cols:
        if row[col_id] in rejected_ids or pd.isna(row[col_id]):
            continue

        if row[col_resp] == "Very unsure":
            val_counts[idx, 0] += 1
        elif row[col_resp] == "Somewhat unsure":
            val_counts[idx, 1] += 1
        elif row[col_resp] == "Neutral":
            val_counts[idx, 2] += 1
        elif row[col_resp] == "Somewhat confident":
            val_counts[idx, 3] += 1
        else:
            val_counts[idx, 4] += 1    


In [149]:
val_counts[1:20,:]

array([[0., 0., 0., 2., 5.],
       [1., 0., 3., 3., 0.],
       [0., 0., 0., 3., 4.],
       [0., 0., 5., 2., 0.],
       [0., 0., 1., 5., 1.],
       [0., 0., 2., 4., 1.],
       [0., 0., 0., 4., 3.],
       [0., 0., 0., 5., 2.],
       [0., 1., 0., 5., 1.],
       [0., 1., 0., 5., 1.],
       [0., 0., 1., 3., 3.],
       [0., 0., 3., 4., 1.],
       [1., 1., 0., 3., 2.],
       [0., 1., 0., 4., 2.],
       [0., 0., 1., 3., 3.],
       [0., 1., 0., 4., 2.],
       [0., 0., 2., 4., 1.],
       [0., 1., 1., 2., 3.],
       [0., 1., 1., 3., 2.]])

In [148]:
krippendorff.alpha(value_counts=val_counts, level_of_measurement="nominal")

np.float64(0.03940373734443603)

In [47]:
annotator_cols = [col for col in results_df.columns if col.endswith('_Response')]

# Only consider rows where the Question matches the target
target_rows = results_df["Question"] == "Based only on the debate transcripts, which Speaker A presented the more persuasive arguments?"

def count_sample1(row):
    return sum([
        1 if val == "Speaker A from Debate 1" else
        0.5 if val == "Both Speaker As were equally persuasive" else
        0
        for val in row
    ])

def count_sample2(row):
    return sum([
        1 if val == "Speaker A from Debate 2" else
        0.5 if val == "Both Speaker As were equally persuasive" else
        0
        for val in row
    ])

results_df.loc[target_rows, 'sample1_win_count'] = results_df.loc[target_rows, annotator_cols].apply(count_sample1, axis=1)
results_df.loc[target_rows, 'sample2_win_count'] = results_df.loc[target_rows, annotator_cols].apply(count_sample2, axis=1)

results_df.head()

,DataPoint_ID,Task_Group_ID,Task_Type,Question,Debate Topic,Debate 1,Debate 2,META_SAMPLE1_POLITICAL_POSITION,META_SAMPLE1_SPEAKERA,META_COMPARISONID_1,META_SAMPLE1_SPEAKERB,META_COMPARISON,META_TASK_GROUP_ID,META_COMPARISONID,META_ATTEN_CHECK_PERSUASIVENESS_ANSWER,META_ATTEN_CHECK_CONFIDENCE_ANSWER,META_SAMPLE1_DIALOGUE_CHUNK,META_PAIR_SESSION_IDS,META_SAMPLE2_POLITICAL_POSITION,Annotator1_ID,Annotator1_Response,Annotator1_Timestamp,Annotator2_ID,Annotator2_Response,Annotator2_Timestamp,Annotator3_ID,Annotator3_Response,Annotator3_Timestamp,Annotator4_ID,Annotator4_Response,Annotator4_Timestamp,Annotator5_ID,Annotator5_Response,Annotator5_Timestamp,Annotator6_ID,Annotator6_Response,Annotator6_Timestamp,Annotator7_ID,Annotator7_Response,Annotator7_Timestamp,Annotator8_ID,Annotator8_Response,Annotator8_Timestamp,sample1_win_count,sample2_win_count
0,019c5684-b730-70dd-b5f3-ca369d6a622c,412,multiple_choice,"Based only on the debate transcripts, which Sp...",Debate the trade-offs of government interventi...,"Speaker A: ""How can this be done if the opport...","Speaker A: ""Would you concur that this point r...",left,human_9,1154,gpt-4o_base,"('human_9', 'human_11')",412,2064,NaN,NaN,"[{'speaker': 'human_9', 'timestamp': '2026-01-...","('XZV57F', 'OR2XBB')",left,6318acc6273e916b39a95a85,Speaker A from Debate 2,2026-02-13T10:46:55.851Z,579f55486475d400015ab683,Speaker A from Debate 1,2026-02-13T12:06:32.977Z,5e90781c06c6ee000974d087,Speaker A from Debate 1,2026-02-13T19:45:23.867Z,66066cf2a8876c1d1907732a,Speaker A from Debate 1,2026-02-14T20:40:58.661Z,5d07689bad18d40001bb73c1,Speaker A from Debate 1,2026-02-16T15:57:15.704Z,63d13c076dd249ca3a6bffbb,Speaker A from Debate 1,2026-02-18T22:06:51.931Z,5f0c94983314760efb2814c1,Speaker A from Debate 1,2026-02-21T11:14:56.612Z,NaN,NaN,NaN,6.0,1.0
4,019c5684-bc10-73fa-ae9c-bdda3c2e3812,417,multiple_choice,"Based only on the debate transcripts, which Sp...",Debate the trade-offs of government interventi...,"Speaker B: ""Market forces ensure that the best...","Speaker A: ""Government intervention in public ...",left,gpt-4o_mas_rag,763,mistral-medium_mas,"('gpt-4o_mas_rag', 'human_6')",417,2089,NaN,NaN,"[{'speaker': 'mistral-medium_mas', 'timestamp'...","('gpt-4o_mas_rag:l__vs__mistral-medium_mas:r',...",left,5dd9a7ff98981d926df9ea18,Speaker A from Debate 1,2026-02-13T10:45:34.049Z,6761658b4037ac25f8afbd22,Speaker A from Debate 1,2026-02-13T11:48:57.224Z,5e2c8f69cad44b2bc5ce9a34,Speaker A from Debate 1,2026-02-13T14:38:04.633Z,67605337c41b9ba5ae52a0c8,Speaker A from Debate 1,2026-02-13T21:03:16.757Z,673f069497dc06b5ecbf7470,Speaker A from Debate 1,2026-02-15T11:47:01.602Z,6658ed9e2802dccba9041ca4,Speaker A from Debate 1,2026-02-17T10:05:08.725Z,677d4d04da6aea9117c478b5,Speaker A from Debate 1,2026-02-20T11:56:31.327Z,NaN,NaN,NaN,7.0,0.0
8,019c5683-7bc9-737c-98f5-9d58923d45e6,60,multiple_choice,"Based only on the debate transcripts, which Sp...",Debate the trade-offs of government interventi...,"Speaker A: ""Resentment and alienisation within...","Speaker A: ""Are taxes the only way to develop ...",right,human_6,988,grok-3_mas,"('human_6', 'human_12')",60,304,NaN,NaN,"[{'speaker': 'human_6', 'timestamp': '2026-01-...","('LU5JFB', 'VIJPFB')",right,5e908233a1ea4d013e973b75,Both Speaker As were equally persuasive,2026-02-13T11:15:52.377Z,653d14bc14d143d888af01b0,Speaker A from Debate 2,2026-02-13T13:26:19.093Z,67be393ad49f9deccf6ddd22,Speaker A from Debate 2,2026-02-13T17:26:28.900Z,5e64da5ac34e832466061bf8,Speaker A from Debate 2,2026-02-14T12:59:43.415Z,671f96e923e76ff949ad6cbc,Both Speaker As were equally persuasive,2026-02-16T10:12:34.555Z,60ff1c49e50b11c8d19c33dc,Speaker A from Debate 1,2026-02-18T12:35:04.636Z,569a54a5217ca2000cc93db0,Both Speaker As were equally persuasive,2026-02-20T19:46:57.078Z,NaN,NaN,NaN,2.5,4.5
10,019c5684-b2c9-734b-a917-fa2e76eeaff7,408,multiple_choice,"Based only on the debate transcripts, which Sp...",Debate the trade-offs of government interventi...,"Speaker A: 

In [48]:
for col in ["average_confidence", "std_confidence", "confidence_values"]:
    if col in results_df.columns:
        results_df.drop(col, axis=1, inplace=True)

In [49]:
likert_map = {
        "Very unsure": 1,
            "Somewhat unsure": 2,
                "Neutral": 3,
                    "Somewhat confident": 4,
                        "Very confident": 5
                        }


# Select annotator response columns
annotator_cols = [col for col in conf_df.columns if col.endswith('_Response')]

# Map likert responses to numbers
conf_df_numeric = conf_df[annotator_cols].replace(likert_map)

# Create a column with list of confidence values (excluding NaN)
conf_df["confidence_values"] = conf_df_numeric.apply(
    lambda row: [v for v in row if pd.notna(v)], axis=1
)

# Compute mean and std for each row in conf_df
conf_df["average_confidence"] = conf_df_numeric.mean(axis=1)
conf_df["std_confidence"] = conf_df_numeric.std(axis=1)

# Merge confidence columns into results_df by comparison ID
results_df = results_df.merge(
    conf_df[["META_COMPARISONID", "average_confidence", "std_confidence", "confidence_values"]],
    on="META_COMPARISONID",
    how="left"
)

results_df.head()

,DataPoint_ID,Task_Group_ID,Task_Type,Question,Debate Topic,Debate 1,Debate 2,META_SAMPLE1_POLITICAL_POSITION,META_SAMPLE1_SPEAKERA,META_COMPARISONID_1,META_SAMPLE1_SPEAKERB,META_COMPARISON,META_TASK_GROUP_ID,META_COMPARISONID,META_ATTEN_CHECK_PERSUASIVENESS_ANSWER,META_ATTEN_CHECK_CONFIDENCE_ANSWER,META_SAMPLE1_DIALOGUE_CHUNK,META_PAIR_SESSION_IDS,META_SAMPLE2_POLITICAL_POSITION,Annotator1_ID,Annotator1_Response,Annotator1_Timestamp,Annotator2_ID,Annotator2_Response,Annotator2_Timestamp,Annotator3_ID,Annotator3_Response,Annotator3_Timestamp,Annotator4_ID,Annotator4_Response,Annotator4_Timestamp,Annotator5_ID,Annotator5_Response,Annotator5_Timestamp,Annotator6_ID,Annotator6_Response,Annotator6_Timestamp,Annotator7_ID,Annotator7_Response,Annotator7_Timestamp,Annotator8_ID,Annotator8_Response,Annotator8_Timestamp,sample1_win_count,sample2_win_count,average_confidence,std_confidence,confidence_values
0,019c5684-b730-70dd-b5f3-ca369d6a622c,412,multiple_choice,"Based only on the debate transcripts, which Sp...",Debate the trade-offs of government interventi...,"Speaker A: ""How can this be done if the opport...","Speaker A: ""Would you concur that this point r...",left,human_9,1154,gpt-4o_base,"('human_9', 'human_11')",412,2064,NaN,NaN,"[{'speaker': 'human_9', 'timestamp': '2026-01-...","('XZV57F', 'OR2XBB')",left,6318acc6273e916b39a95a85,Speaker A from Debate 2,2026-02-13T10:46:55.851Z,579f55486475d400015ab683,Speaker A from Debate 1,2026-02-13T12:06:32.977Z,5e90781c06c6ee000974d087,Speaker A from Debate 1,2026-02-13T19:45:23.867Z,66066cf2a8876c1d1907732a,Speaker A from Debate 1,2026-02-14T20:40:58.661Z,5d07689bad18d40001bb73c1,Speaker A from Debate 1,2026-02-16T15:57:15.704Z,63d13c076dd249ca3a6bffbb,Speaker A from Debate 1,2026-02-18T22:06:51.931Z,5f0c94983314760efb2814c1,Speaker A from Debate 1,2026-02-21T11:14:56.612Z,NaN,NaN,NaN,6.0,1.0,3.714286,1.112697,"[3, 4, 5, 2, 3, 5, 4]"
1,019c5684-bc10-73fa-ae9c-bdda3c2e3812,417,multiple_choice,"Based only on the debate transcripts, which Sp...",Debate the trade-offs of government interventi...,"Speaker B: ""Market forces ensure that the best...","Speaker A: ""Government intervention in public ...",left,gpt-4o_mas_rag,763,mistral-medium_mas,"('gpt-4o_mas_rag', 'human_6')",417,2089,NaN,NaN,"[{'speaker': 'mistral-medium_mas', 'timestamp'...","('gpt-4o_mas_rag:l__vs__mistral-medium_mas:r',...",left,5dd9a7ff98981d926df9ea18,Speaker A from Debate 1,2026-02-13T10:45:34.049Z,6761658b4037ac25f8afbd22,Speaker A from Debate 1,2026-02-13T11:48:57.224Z,5e2c8f69cad44b2bc5ce9a34,Speaker A from Debate 1,2026-02-13T14:38:04.633Z,67605337c41b9ba5ae52a0c8,Speaker A from Debate 1,2026-02-13T21:03:16.757Z,673f069497dc06b5ecbf7470,Speaker A from Debate 1,2026-02-15T11:47:01.602Z,6658ed9e2802dccba9041ca4,Speaker A from Debate 1,2026-02-17T10:05:08.725Z,677d4d04da6aea9117c478b5,Speaker A from Debate 1,2026-02-20T11:56:31.327Z,NaN,NaN,NaN,7.0,0.0,4.714286,0.48795,"[5, 4, 4, 5, 5, 5, 5]"
2,019c5683-7bc9-737c-98f5-9d58923d45e6,60,multiple_choice,"Based only on the debate transcripts, which Sp...",Debate the trade-offs of government interventi...,"Speaker A: ""Resentment and alienisation within...","Speaker A: ""Are taxes the only way to develop ...",right,human_6,988,grok-3_mas,"('human_6', 'human_12')",60,304,NaN,NaN,"[{'speaker': 'human_6', 'timestamp': '2026-01-...","('LU5JFB', 'VIJPFB')",right,5e908233a1ea4d013e973b75,Both Speaker As were equally persuasive,2026-02-13T11:15:52.377Z,653d14bc14d143d888af01b0,Speaker A from Debate 2,2026-02-13T13:26:19.093Z,67be393ad49f9deccf6ddd22,Speaker A from Debate 2,2026-02-13T17:26:28.900Z,5e64da5ac34e832466061bf8,Speaker A from Debate 2,2026-02-14T12:59:43.415Z,671f96e923e76ff949ad6cbc,Both Speaker As were equally persuasive,2026-02-16T10:12:34.555Z,60ff1c49e50b11c8d19c33dc,Speaker A from Debate 1,2026-02-18T12:35:04.636Z,569a54a5217ca2000cc93db0,Both Speaker As were equally persuasive,2026-02-20T19:46:57.078Z,NaN,NaN,NaN,2.5,4.5,3.142857,1.069045,"[4, 3, 4, 4, 3, 3, 1

In [50]:
comparison_ids = df["META_COMPARISONID"].unique().tolist()

annotator_cols = []
for col in df.columns:
    if col.startswith("Annotator") and col.endswith('_ID'):
        base = col.replace("_ID", "")
        annotator_cols.append((
            col,                          # ID column
            base + "_Timestamp",          # Timestamp column
            base + "_Response"            # Response column
        ))
annotator_cols

[('Annotator1_ID', 'Annotator1_Timestamp', 'Annotator1_Response'),
 ('Annotator2_ID', 'Annotator2_Timestamp', 'Annotator2_Response'),
 ('Annotator3_ID', 'Annotator3_Timestamp', 'Annotator3_Response'),
 ('Annotator4_ID', 'Annotator4_Timestamp', 'Annotator4_Response'),
 ('Annotator5_ID', 'Annotator5_Timestamp', 'Annotator5_Response'),
 ('Annotator6_ID', 'Annotator6_Timestamp', 'Annotator6_Response'),
 ('Annotator7_ID', 'Annotator7_Timestamp', 'Annotator7_Response'),
 ('Annotator8_ID', 'Annotator8_Timestamp', 'Annotator8_Response')]

In [51]:
annotator_cols = [y for x in annotator_cols for y in x]
annotator_cols

['Annotator1_ID',
 'Annotator1_Timestamp',
 'Annotator1_Response',
 'Annotator2_ID',
 'Annotator2_Timestamp',
 'Annotator2_Response',
 'Annotator3_ID',
 'Annotator3_Timestamp',
 'Annotator3_Response',
 'Annotator4_ID',
 'Annotator4_Timestamp',
 'Annotator4_Response',
 'Annotator5_ID',
 'Annotator5_Timestamp',
 'Annotator5_Response',
 'Annotator6_ID',
 'Annotator6_Timestamp',
 'Annotator6_Response',
 'Annotator7_ID',
 'Annotator7_Timestamp',
 'Annotator7_Response',
 'Annotator8_ID',
 'Annotator8_Timestamp',
 'Annotator8_Response']

In [55]:
for col in annotator_cols:
    if col in results_df.columns:
        results_df.drop(col, axis=1, inplace=True)

remove_cols = ["META_ATTEN_CHECK_PERSUASIVENESS_ANSWER","META_ATTEN_CHECK_CONFIDENCE_ANSWER"]
for col in remove_cols:
    if col in results_df.columns:
        results_df.drop(col, axis=1, inplace=True)

In [56]:
results_df.head()

,DataPoint_ID,Task_Group_ID,Task_Type,Question,Debate Topic,Debate 1,Debate 2,META_SAMPLE1_POLITICAL_POSITION,META_SAMPLE1_SPEAKERA,META_COMPARISONID_1,META_SAMPLE1_SPEAKERB,META_COMPARISON,META_TASK_GROUP_ID,META_COMPARISONID,META_SAMPLE1_DIALOGUE_CHUNK,META_PAIR_SESSION_IDS,META_SAMPLE2_POLITICAL_POSITION,sample1_win_count,sample2_win_count,average_confidence,std_confidence,confidence_values
0,019c5684-b730-70dd-b5f3-ca369d6a622c,412,multiple_choice,"Based only on the debate transcripts, which Sp...",Debate the trade-offs of government interventi...,"Speaker A: ""How can this be done if the opport...","Speaker A: ""Would you concur that this point r...",left,human_9,1154,gpt-4o_base,"('human_9', 'human_11')",412,2064,"[{'speaker': 'human_9', 'timestamp': '2026-01-...","('XZV57F', 'OR2XBB')",left,6.0,1.0,3.714286,1.112697,"[3, 4, 5, 2, 3, 5, 4]"
1,019c5684-bc10-73fa-ae9c-bdda3c2e3812,417,multiple_choice,"Based only on the debate transcripts, which Sp...",Debate the trade-offs of government interventi...,"Speaker B: ""Market forces ensure that the best...","Speaker A: ""Government intervention in public ...",left,gpt-4o_mas_rag,763,mistral-medium_mas,"('gpt-4o_mas_rag', 'human_6')",417,2089,"[{'speaker': 'mistral-medium_mas', 'timestamp'...","('gpt-4o_mas_rag:l__vs__mistral-medium_mas:r',...",left,7.0,0.0,4.714286,0.48795,"[5, 4, 4, 5, 5, 5, 5]"
2,019c5683-7bc9-737c-98f5-9d58923d45e6,60,multiple_choice,"Based only on the debate transcripts, which Sp...",Debate the trade-offs of government interventi...,"Speaker A: ""Resentment and alienisation within...","Speaker A: ""Are taxes the only way to develop ...",right,human_6,988,grok-3_mas,"('human_6', 'human_12')",60,304,"[{'speaker': 'human_6', 'timestamp': '2026-01-...","('LU5JFB', 'VIJPFB')",right,2.5,4.5,3.142857,1.069045,"[4, 3, 4, 4, 3, 3, 1]"
3,019c5684-b2c9-734b-a917-fa2e76eeaff7,408,multiple_choice,"Based only on the debate transcripts, which Sp...",Debate the trade-offs of government interventi...,"Speaker A: ""Governments often struggle to meas...","Speaker B: ""Unlike the NHS, which leaves the m...",right,human_10,899,gpt-4o_base,"('human_10', 'grok-3_mas_rag')",408,2040,"[{'speaker': 'human_10', 'timestamp': '2026-01...","('7BKBI8', 'grok-3_mas_rag:r__vs__grok-3_mas:l')",right,0.0,7.0,4.571429,0.534522,"[5, 5, 4, 5, 4, 4, 5]"
4,019c5683-84d1-75cf-b5fa-cb13c872e2b1,71,multiple_choice,"Based only on the debate transcripts, which Sp...",Debate the trade-offs of government interventi...,"Speaker B: ""Did you see the current job market...","Speaker A: ""Scandinavian countries have implem...",left,human_4,1004,human_13,"('human_4', 'human_2')",71,356,"[{'speaker': 'human_13', 'timestamp': '2026-01...","('KNI0QV', 'X12MFO')",left,6.0,1.0,3.285714,0.48795,"[3, 3, 3, 4, 3, 3, 4]"


In [54]:
results_df.to_csv("data/subjective-persuasiveness-prolific-results.csv", sep="|")